# Notebook 01 — Modelos LLM e NLP com Hugging Face

**Objetivo:** Demonstrar dominio do ecossistema Hugging Face com tarefas NLP aplicadas ao dominio de bulas medicas, seguindo o estilo do professor (`pipeline`, `AutoTokenizer`, `AutoModel`).

**Rubrica 1:** Construir aplicacoes NLP com LLMs e ecossistema Hugging Face (5 itens).

## 2.1 Setup e Imports

In [1]:
import torch
from transformers import pipeline, AutoModel, AutoTokenizer
from scripts.config import DEVICE, NER_MODEL, EMBEDDING_MODEL

print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponivel: {torch.cuda.is_available()}")
print(f"Device configurado: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch: 2.6.0+cu124
CUDA disponivel: True
Device configurado: cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
VRAM: 6.4 GB


## 2.2 Carregando Modelo com AutoModel + AutoTokenizer

Demonstracao no estilo do professor: carregar um modelo pre-treinado, tokenizar entrada, e inspecionar as dimensoes dos hidden states.

**Modelo:** `pucpr/clinicalnerpt-chemical` — BERT treinado para NER em textos clinicos em portugues.

### Por que comecar com AutoModel?

`AutoModel.from_pretrained()` carrega o corpo do modelo (encoder) **sem cabecalho de tarefa** — util para entender a arquitetura antes de adicionar classificadores. As dimensoes `[Batch, Tokens, Hidden_Dim]` revelam:
- **Batch:** quantas frases processadas de uma vez
- **Tokens:** quantos tokens a tokenizacao gerou (incluindo `[CLS]` e `[SEP]`)
- **Hidden_Dim:** tamanho do embedding interno (768 para BERT base)

In [2]:
model_id = NER_MODEL  # "pucpr/clinicalnerpt-chemical"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModel.from_pretrained(model_id).to(DEVICE)

# Processando a entrada (estilo do professor)
inputs = tokenizer("O mecanismo de atencao e poderoso", return_tensors="pt")
# Move para GPU
inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
outputs = model(**inputs)

print(f"Dimensoes do output: {outputs.last_hidden_state.shape}")
print(f"Interpretacao: [Batch={outputs.last_hidden_state.shape[0]}, Tokens={outputs.last_hidden_state.shape[1]}, Hidden_Dim={outputs.last_hidden_state.shape[2]}]")

# Mostrar os tokens gerados
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print(f"\nTokens: {tokens}")
print(f"Total de tokens: {len(tokens)}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: pucpr/clinicalnerpt-chemical
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/709M [00:00<?, ?B/s]

Dimensoes do output: torch.Size([1, 10, 768])
Interpretacao: [Batch=1, Tokens=10, Hidden_Dim=768]

Tokens: ['[CLS]', 'o', 'mecanismo', 'de', 'at', '##en', '##cao', 'e', 'poderoso', '[SEP]']
Total de tokens: 10


**Observacoes:**
- O tokenizador BERT usa **WordPiece**: palavras frequentes viram tokens unicos, palavras raras sao quebradas em sub-tokens
- O limite padrao e **512 tokens** — textos mais longos precisam de estrategias de truncamento ou chunking
- `last_hidden_state` contem o embedding contextualizado de cada token — diferente de embeddings estaticos (Word2Vec), estes variam conforme o contexto da frase

## 2.3 Pipeline: sentiment-analysis em Frases Clinicas

Usando o pipeline default de sentiment-analysis do Hugging Face para classificar frases extraidas de bulas medicas. O objetivo nao e obter resultados perfeitos, mas **demonstrar as limitacoes** de um modelo generico em dominio especializado — o que motiva o fine-tuning posterior.

In [3]:
classifier = pipeline("sentiment-analysis")

# Frases reais de bulas medicas
frases = [
    "O uso concomitante e contraindicado devido ao risco de arritmia fatal.",
    "Nao ha interacoes conhecidas com este medicamento.",
    "Recomenda-se monitoramento da funcao renal durante o tratamento.",
    "A administracao concomitante de Amoxicilina com Metotrexato pode aumentar a toxicidade.",
    "O medicamento e seguro e bem tolerado pela maioria dos pacientes.",
]

for frase in frases:
    resultado = classifier(frase)[0]
    print(f"[{resultado["label"]:>8} | {resultado["score"]:.3f}] {frase}")

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

C:\workspace\python\projeto-2-modulo-1-pos\venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\lord_\.cache\huggingface\hub\models--distilbert--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

[NEGATIVE | 0.991] O uso concomitante e contraindicado devido ao risco de arritmia fatal.
[NEGATIVE | 0.983] Nao ha interacoes conhecidas com este medicamento.
[NEGATIVE | 0.905] Recomenda-se monitoramento da funcao renal durante o tratamento.
[NEGATIVE | 0.934] A administracao concomitante de Amoxicilina com Metotrexato pode aumentar a toxicidade.
[POSITIVE | 0.543] O medicamento e seguro e bem tolerado pela maioria dos pacientes.


**Analise:**

- O modelo generico classifica frases como POSITIVE/NEGATIVE com base no tom emocional, **nao no significado clinico**
- Frase com "contraindicado" e "fatal" → NEGATIVE (correto por acaso)
- Frase com "nao ha interacoes" → POSITIVE (correto por acaso)
- **Limitacao:** "aumentar a toxicidade" pode ser classificado como NEGATIVE pelo tom, mas o modelo nao entende que isso descreve uma **interacao medicamentosa grave**
- **Conclusao:** Precisamos de modelos treinados em dominio clinico (como o `clinicalnerpt-chemical`) e fine-tuning especifico para classificacao de interacoes